In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
# =========================
# IMPORTS
# =========================
import os
import cv2
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader

# =========================
# MOUNT DRIVE
# =========================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# =========================
# UNZIP DATASET
# =========================
import zipfile

zip_path = "/content/drive/MyDrive/AVEC2014.zip"
extract_path = "/content/avec2014"

if not os.path.exists(extract_path + "/AVEC2014"):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)

print("Dataset ready at:", extract_path)

# =========================
# LOAD LABELS (CONSISTENT KEYS)
# =========================
def load_labels():
    df = pd.read_csv("/content/avec2014/AVEC2014/labels.csv")

    labels = {}
    for _, row in df.iterrows():
        key = str(row["filename"]).strip()
        key = key.replace("\\", "/")

        # normalize keys
        key = key.replace("Training/", "")
        key = key.replace("Testing/", "")

        key = os.path.splitext(key)[0]

        labels[key] = float(row["BDI-II"])

    print("Labels loaded:", len(labels))
    return labels


# =========================
# TRAINING DATASET (SLIDING WINDOW)
# =========================
class AVECDatasetTrain(Dataset):
    def __init__(self, video_dir, labels, clip_len=16, stride=8, max_clips_per_video=5):
        self.samples = []
        self.clip_len = clip_len

        for root, _, files in os.walk(video_dir):
            for f in files:
                if not f.endswith(".mp4"):
                    continue

                path = os.path.join(root, f)

                # SAME KEY LOGIC
                key = path.split("AVEC2014/")[-1]

                if key.startswith("Training/"):
                    key = key.replace("Training/", "")
                elif key.startswith("Testing/"):
                    key = key.replace("Testing/", "")

                key = os.path.splitext(key)[0]
                key = key.replace("\\", "/").strip()

                if key not in labels:
                    continue

                label = labels[key]

                cap = cv2.VideoCapture(path)
                total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
                cap.release()

                if total_frames < clip_len:
                    continue

                starts = list(range(0, total_frames - clip_len + 1, stride))

                # LIMIT clips (keeps training fast)
                if len(starts) > max_clips_per_video:
                    starts = np.random.choice(starts, max_clips_per_video, replace=False)

                for s in starts:
                    self.samples.append((path, int(s), label))

        print("Total training clips:", len(self.samples))

    def load_clip(self, path, start):
        cap = cv2.VideoCapture(path)
        cap.set(cv2.CAP_PROP_POS_FRAMES, start)

        frames = []

        for _ in range(self.clip_len):
            ret, frame = cap.read()
            if not ret:
                break

            frame = cv2.resize(frame, (112, 112))
            frames.append(frame)

        cap.release()

        if len(frames) < self.clip_len:
            return None

        frames = np.array(frames)
        frames = np.transpose(frames, (3, 0, 1, 2)) / 255.0

        return torch.tensor(frames, dtype=torch.float32)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, start, label = self.samples[idx]

        clip = self.load_clip(path, start)

        if clip is None:
            return self.__getitem__((idx + 1) % len(self.samples))

        return clip, torch.tensor(label, dtype=torch.float32)


# =========================
# EVALUATION DATASET (FULL VIDEO)
# =========================
class AVECDatasetEval(Dataset):
    def __init__(self, video_dir, labels):
        self.samples = []

        for root, _, files in os.walk(video_dir):
            for f in files:
                if not f.endswith(".mp4"):
                    continue

                path = os.path.join(root, f)

                # SAME KEY LOGIC
                key = path.split("AVEC2014/")[-1]

                if key.startswith("Training/"):
                    key = key.replace("Training/", "")
                elif key.startswith("Testing/"):
                    key = key.replace("Testing/", "")

                key = os.path.splitext(key)[0]
                key = key.replace("\\", "/").strip()

                if key in labels:
                    self.samples.append((path, labels[key]))

        print("Total evaluation videos:", len(self.samples))

    def load_video(self, path):
        cap = cv2.VideoCapture(path)
        frames = []

        while True:
            ret, frame = cap.read()
            if not ret:
                break

            frame = cv2.resize(frame, (112, 112))
            frames.append(frame)

        cap.release()

        if len(frames) == 0:
            return None

        frames = np.array(frames)
        frames = np.transpose(frames, (3, 0, 1, 2)) / 255.0

        return torch.tensor(frames, dtype=torch.float32)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]

        video = self.load_video(path)

        if video is None:
            return self.__getitem__((idx + 1) % len(self.samples))

        return video, torch.tensor(label, dtype=torch.float32)


# =========================
# INITIALIZE DATASETS
# =========================
labels = load_labels()

# 🔴 TRAINING
train_dataset = AVECDatasetTrain(
    "/content/avec2014/AVEC2014/Training",
    labels,
    clip_len=16,
    stride=8,
    max_clips_per_video=5
)

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)

# 🟢 EVALUATION
eval_dataset = AVECDatasetEval(
    "/content/avec2014/AVEC2014/Testing",
    labels
)

print("Pipeline ready.")

Mounted at /content/drive
Dataset ready at: /content/avec2014
Labels loaded: 300
Total training clips: 500
Total evaluation videos: 100
Pipeline ready.


In [ ]:
# =========================
# IMPORTS
# =========================
import torch
import torch.nn as nn
from google.colab import drive

# =========================
# MOUNT DRIVE
# =========================
drive.mount('/content/drive', force_remount=True)

checkpoint_path = "/content/drive/MyDrive/c3d_checkpoint_stage1.pth"
best_model_path = "/content/drive/MyDrive/best_c3d_stage1.pth"

# =========================
# MODEL (C3D)
# =========================
class C3D(nn.Module):
    def __init__(self):
        super(C3D, self).__init__()

        self.features = nn.Sequential(
            nn.Conv3d(3, 64, 3, padding=1), nn.ReLU(), nn.MaxPool3d(2),
            nn.Conv3d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool3d(2),

            nn.Conv3d(128, 256, 3, padding=1), nn.ReLU(),
            nn.Conv3d(256, 256, 3, padding=1), nn.ReLU(),
            nn.MaxPool3d(2),

            nn.Conv3d(256, 512, 3, padding=1), nn.ReLU(),
            nn.Conv3d(512, 512, 3, padding=1), nn.ReLU(),
            nn.MaxPool3d(2)
        )

        self.classifier = nn.Sequential(
            nn.Linear(512 * 1 * 7 * 7, 4096),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(4096, 4096),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(4096, 1)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

# =========================
# DEVICE
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = C3D().to(device)

# =========================
# LOAD PRETRAINED WEIGHTS (IMPORTANT BOOST)
# =========================
try:
    pretrained_path = "/content/drive/MyDrive/c3d-pretrained.pth"

    state_dict = torch.load(pretrained_path, map_location=device)

    model_dict = model.state_dict()

    # Filter matching layers (ignore classifier)
    pretrained_dict = {
        k: v for k, v in state_dict.items()
        if k in model_dict and "classifier" not in k
    }

    model_dict.update(pretrained_dict)
    model.load_state_dict(model_dict)

    print(f"✅ Pretrained weights loaded: {len(pretrained_dict)} layers matched")

except Exception as e:
    print("❌ Failed to load pretrained weights:", e)

# =========================
# DATASET FROM PREVIOUS CELL
# =========================
# uses: train_dataset, train_loader (already created)

loader = train_loader

# =========================
# STAGE 1 FREEZE
# =========================
for param in model.features.parameters():
    param.requires_grad = False

print("Stage 1: Feature extractor frozen.")

# =========================
# OPTIMIZER
# =========================
criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=5e-5
)

# =========================
# TRAINING (FIXED)
# =========================
def train():

    model.train()

    best_loss = float("inf")
    patience = 10   # FIXED (was too small)
    counter = 0

    for epoch in range(20):

        total_loss = 0

        for videos, targets in loader:
            videos = videos.to(device)
            targets = targets.to(device).view(-1, 1)

            preds = model(videos)
            loss = criterion(preds, targets)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(loader)

        print(f"[Stage 1] Epoch {epoch+1} | Loss: {avg_loss:.4f}")

        torch.save(model.state_dict(), checkpoint_path)

        if avg_loss < best_loss:
            best_loss = avg_loss
            counter = 0

            torch.save(model.state_dict(), best_model_path)
            print("Best model saved.")

        else:
            counter += 1
            print(f"No improvement ({counter}/{patience})")

        if counter >= patience:
            print("Early stopping triggered.")
            break


# =========================
# RUN STAGE 1
# =========================
train()

Mounted at /content/drive
✅ Pretrained weights loaded: 12 layers matched
Stage 1: Feature extractor frozen.
[Stage 1] Epoch 1 | Loss: 113.5073
Best model saved.
[Stage 1] Epoch 2 | Loss: 41.3419
Best model saved.
[Stage 1] Epoch 3 | Loss: 25.2410
Best model saved.
[Stage 1] Epoch 4 | Loss: 17.3629
Best model saved.
[Stage 1] Epoch 5 | Loss: 11.2667
Best model saved.
[Stage 1] Epoch 6 | Loss: 10.3896
Best model saved.
[Stage 1] Epoch 7 | Loss: 8.6164
Best model saved.
[Stage 1] Epoch 8 | Loss: 8.2713
Best model saved.
[Stage 1] Epoch 9 | Loss: 8.3991
No improvement (1/10)
[Stage 1] Epoch 10 | Loss: 8.1551
Best model saved.
[Stage 1] Epoch 11 | Loss: 9.3356
No improvement (1/10)
[Stage 1] Epoch 12 | Loss: 7.4793
Best model saved.
[Stage 1] Epoch 13 | Loss: 7.7598
No improvement (1/10)
[Stage 1] Epoch 14 | Loss: 7.8992
No improvement (2/10)


KeyboardInterrupt: 

In [ ]:
# =========================
# STAGE 2: FINE-TUNING (UNFREEZE)
# =========================

print("Starting Stage 2 fine-tuning...")

# Load best Stage 1 model
model.load_state_dict(torch.load(best_model_path, map_location=device))

# =========================
# UNFREEZE ALL LAYERS
# =========================
for param in model.parameters():
    param.requires_grad = True

print("All layers unfrozen for fine-tuning.")

# =========================
# LOWER LEARNING RATE
# =========================
optimizer = torch.optim.Adam(model.parameters(), lr=5e-6)
criterion = nn.MSELoss()

# =========================
# TRAINING LOOP (STAGE 2)
# =========================
def train_stage2():

    model.train()

    best_loss = float("inf")
    patience = 10
    counter = 0

    for epoch in range(30):   # slightly reduced

        total_loss = 0

        for videos, targets in loader:
            videos = videos.to(device)
            targets = targets.to(device).view(-1, 1)

            preds = model(videos)
            loss = criterion(preds, targets)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(loader)

        print(f"[Stage 2] Epoch {epoch+1} | Loss: {avg_loss:.4f}")

        # =========================
        # SAVE BEST MODEL ONLY
        # =========================
        if avg_loss < best_loss:
            best_loss = avg_loss
            counter = 0

            torch.save(
                model.state_dict(),
                "/content/drive/MyDrive/best_c3d_stage2.pth"
            )
            print("Stage 2 best model saved.")

        else:
            counter += 1
            print(f"No improvement ({counter}/{patience})")

        if counter >= patience:
            print("Stage 2 early stopping triggered.")
            break

# =========================
# RUN STAGE 2
# =========================
train_stage2()

Starting Stage 2 fine-tuning...
All layers unfrozen for fine-tuning.
[Stage 2] Epoch 1 | Loss: 7.2959
Stage 2 best model saved.
[Stage 2] Epoch 2 | Loss: 6.0889
Stage 2 best model saved.
[Stage 2] Epoch 3 | Loss: 4.8221
Stage 2 best model saved.
[Stage 2] Epoch 4 | Loss: 4.5152
Stage 2 best model saved.
[Stage 2] Epoch 5 | Loss: 5.0178
No improvement (1/10)
[Stage 2] Epoch 6 | Loss: 5.2487
No improvement (2/10)
[Stage 2] Epoch 7 | Loss: 3.6118
Stage 2 best model saved.
[Stage 2] Epoch 8 | Loss: 3.7144
No improvement (1/10)
[Stage 2] Epoch 9 | Loss: 3.0842
Stage 2 best model saved.
[Stage 2] Epoch 10 | Loss: 3.3794
No improvement (1/10)
[Stage 2] Epoch 11 | Loss: 3.5033
No improvement (2/10)
[Stage 2] Epoch 12 | Loss: 3.1522
No improvement (3/10)
[Stage 2] Epoch 13 | Loss: 3.5094
No improvement (4/10)
[Stage 2] Epoch 14 | Loss: 3.4654
No improvement (5/10)


KeyboardInterrupt: 

In [ ]:
# =========================
# LOAD TRAINED MODEL (FINAL)
# =========================
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Use SAME C3D definition from training
model = C3D().to(device)

# =========================
# LOAD BEST AVAILABLE WEIGHTS
# =========================
stage2_path = "/content/drive/MyDrive/best_c3d_stage2.pth"
stage1_path = "/content/drive/MyDrive/best_c3d_stage1.pth"

if os.path.exists(stage2_path):
    model.load_state_dict(torch.load(stage2_path, map_location=device))
    print("Loaded Stage 2 weights ✅")

elif os.path.exists(stage1_path):
    model.load_state_dict(torch.load(stage1_path, map_location=device))
    print("Loaded Stage 1 weights ⚠️ (Stage 2 missing)")

else:
    raise FileNotFoundError("No trained model weights found.")

model.eval()

# =========================
# QUICK SANITY CHECK
# =========================
total_params = sum(p.numel() for p in model.parameters())
print(f"Model ready | Parameters: {total_params:,}")

Loaded Stage 2 weights ✅
Model ready | Parameters: 133,049,089


In [ ]:
# =========================
# EVALUATION (PAPER-STYLE)
# =========================
import numpy as np
import torch

model.eval()

# =========================
# MULTI-CLIP (SLIDING WINDOW)
# =========================
def get_clips(video_tensor, clip_len=16, stride=8):
    clips = []
    T = video_tensor.shape[1]

    if T < clip_len:
        pad = clip_len - T
        last_frame = video_tensor[:, -1:, :, :]
        padding = last_frame.repeat(1, pad, 1, 1)
        video_tensor = torch.cat([video_tensor, padding], dim=1)
        T = clip_len

    for i in range(0, T - clip_len + 1, stride):
        clips.append(video_tensor[:, i:i+clip_len, :, :])

    return clips


# =========================
# PREDICT SINGLE VIDEO
# =========================
def predict_video(video_tensor):
    clips = get_clips(video_tensor)

    preds = []

    with torch.no_grad():
        for clip in clips:
            clip = clip.unsqueeze(0).to(device)
            pred = model(clip)
            preds.append(pred.item())

    if len(preds) == 0:
        return None

    return float(np.mean(preds))  # explicit


# =========================
# RUN EVALUATION
# =========================
y_true = []
y_pred = []

for video, label in eval_dataset:

    video = video.to(device)  # safety
    pred = predict_video(video)

    if pred is not None:
        y_true.append(label.item())   # FIXED
        y_pred.append(pred)

print("Valid samples used:", len(y_true))

# =========================
# METRICS
# =========================
if len(y_true) == 0:
    print("ERROR: No valid predictions.")
else:
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    # MAE
    mae = np.mean(np.abs(y_true - y_pred))

    # RMSE
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))

    # CCC
    mean_true = np.mean(y_true)
    mean_pred = np.mean(y_pred)

    var_true = np.var(y_true)
    var_pred = np.var(y_pred)

    cov = np.mean((y_true - mean_true) * (y_pred - mean_pred))

    ccc = (2 * cov) / (
        var_true + var_pred + (mean_true - mean_pred) ** 2 + 1e-8
    )

    print("\n===== FINAL PAPER-STYLE RESULTS =====")
    print(f"MAE  : {mae:.4f}")
    print(f"RMSE : {rmse:.4f}")
    print(f"CCC  : {ccc:.4f}")

Valid samples used: 100

===== FINAL PAPER-STYLE RESULTS =====
MAE  : 9.4178
RMSE : 11.5706
CCC  : 0.2755
